<a href="https://colab.research.google.com/github/Racim-dev/MACHINE-LEARNING/blob/main/ENPC_TP5_ClassicML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification of Tree Species with Classic Machine Learning

$\qquad\qquad\qquad\qquad\qquad\qquad\qquad\qquad$
<img src="https://drive.google.com/uc?id=1F4B80Q7XFcMK9aDKXGYRKgKw7q8sRP5i" alt="Cache la Poudre" width="300"/>

<center><em>Cache la Poudre Natural Reserve</em></center>

---

The objective of this practical session is to apply and compare several **classical supervised machine learning algorithms** for a real-world classification task: identifying tree species from environmental descriptors.

The dataset is composed of square forest parcels of size $30 \times 30\,\text{m}^2$.  
Each parcel must be classified into one of the following **seven tree species**:

> **Class index** | **Species**
> --- | ---
> 0 | Spruce / fir
> 1 | Lodgepole pine
> 2 | Ponderosa pine
> 3 | Cottonwood / willow
> 4 | Aspen poplar
> 5 | Douglas-fir
> 6 | Krummholz

For each parcel, we are given **14 handcrafted features**, designed by domain experts and derived from topographic and environmental measurements:

> **Index** | **Unit** | **Description**
> --- | --- | ---
> 0 | meter | Elevation
> 1 | degree | Aspect (azimuth)
> 2 | degree | Slope
> 3 | meter | Horizontal distance to nearest water body
> 4 | meter | Vertical distance to nearest water body
> 5 | meter | Horizontal distance to nearest road
> 6 | index (0–255) | Hillshade at 9am
> 7 | index (0–255) | Hillshade at 12pm
> 8 | index (0–255) | Hillshade at 3pm
> 9 | meter | Horizontal distance to nearest fire ignition point
> 10 | boolean | Rawah Wilderness Area
> 11 | boolean | Neota Wilderness Area
> 12 | boolean | Comanche Peak Wilderness Area
> 13 | boolean | Cache la Poudre Wilderness Area

All code sections that require your intervention are marked with `# TODO`.  
Make sure to complete **all** such sections before moving on.

You are strongly encouraged to:
- create additional cells to inspect variables,
- print tensor shapes,
- experiment with small code snippets.

Debugging is much easier when you verify intermediate results.

<font color='green'>
Q5 and Q8 ARE AT HOME QUESTION: SKIP DURING TP UNTIL YOU ARE FINISHED WITH THE GUIDED QUESTIONS. FOR THE THQ, SEND TO YOUR TA:

- ENTIRE NOTEBOOK
- ANALYSIS FOR Q5 AND Q8 ONLY
- DO NOT SEND A DOCUMENT WILL ALL QUESTIONS CORRECTED IN CLASS, THIS IS NOT USEFUL FOR US.
</font>

---

## Installation and Data Loading




In [ ]:
#import and installations
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn import datasets
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

**Q1**  
Execute the first cells to import to load and explore the dataset.  

In [ ]:
class_names = ["Spruce", "Lodgepole pine", "Ponderosa pine", "Cottonwood", "Aspen poplar", "Douglas-fir", "Krummholz"]
feature_names = ["Elevation", "Azimuth", "Slope", "Hor dist to water", "Vert dist to water",
                 "distance to road", "Hillshade 9am", "Hillshade 12pm", "Hillshade 3pm",
                 "distance to fire", "Rawah Area", "Neota Area", "Comanche Peak Area", "Cache la Poudre Area"]

In [ ]:
X, y = fetch_openml(
    name="covertype",
    version=3,
    as_frame=False,
    return_X_y=True
)

X, _, y, _ = train_test_split(#sample 50k plots for faster computation
    X, y,
    train_size=50000,
    random_state=42,
    stratify=y
)

# Convert labels from {1,...,7} to {0,...,6}
y = y.astype(int) - 1

# Keep only the first 14 features
X = X[:, :14].astype(np.float32)

In [ ]:
# --------------------------------------------------
# Dataset summary
# --------------------------------------------------
print("Dataset summary")
print("----------------")
print(f"Number of samples      : {X.shape[0]}")
print(f"Number of features     : {X.shape[1]}")
print(f"Feature vector shape   : {X.shape}")
print(f"Label vector shape     : {y.shape}")

print("\nExample feature vector (first sample):")
print("------------------------------------")
print(X[0, :])

# --------------------------------------------------
# Label distribution (counts)
# --------------------------------------------------
K = int(y.max()) + 1
counts = np.bincount(y, minlength=K)

# Histogram
plt.figure(figsize=(8, 4))
plt.bar(
    range(K),
    counts,
    tick_label=class_names
)
plt.xlabel("Class")
plt.ylabel("Number of samples")
plt.title("Class distribution")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# --------------------------------------------------
# Pretty table
# --------------------------------------------------
print("\nLabel distribution:")
print("-" * 43)
header_name = "Class name"
print(f"{header_name:>20} | {'Count':>8}")
print("-" * 43)

for k in range(K):
    name = class_names[k] if class_names is not None else str(k)
    print(f"{name:>20} | {counts[k]:>8}")


Q2) We split the dataset into three disjoint subsets:

- a training set used to fit model parameters (60\%),

- a validation set used to tune hyperparameters (20\%),

- a test set used only once for final evaluation (20\%).

Complete the code to split the data into a train+validation set and a test set (20%), then split the remaining data into a training set (60%) and a validation set (20%). What does "stratify" means here, and why is it important?
<font color='red'>we want the same class distribution in all set. Otherwise, the model may overfit the train set distribution, and rare classes may eb entirely absent from the smaller sets!</font>

In [ ]:
# Train / validation / test split
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.25,   # 0.25 × 0.8 = 0.20 of total data
    random_state=42,
    stratify=y_trainval
)

# Prevalence in training set
K = int(max(y_train.max(), y_test.max())) + 1
train_counts = np.bincount(y_train, minlength=K)
train_prevalence = train_counts / train_counts.sum()

print("Dataset split summary")
print("---------------------")
print(f"Training set   : {X_train.shape[0]:>7} samples")
print(f"Validation set : {X_val.shape[0]:>7} samples")
print(f"Test set       : {X_test.shape[0]:>7} samples")

print("\nTotal samples  :", X.shape[0])

assert X_train.shape[0] + X_val.shape[0] + X_test.shape[0] == X.shape[0]
assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1]
assert len(y_train) == X_train.shape[0]

## Q3 — Logistic Regression (Baseline)

We start with **multinomial logistic regression**, a strong baseline for tabular classification.

### Q3.1 — Complete the code to train the regression model. Why do we normalise the input?
<font color='red'>We will standardise features (zero mean, unit variance).  
This improves optimisation because the objective is **better conditioned**: all dimensions have comparable scale, so gradient-based solvers do not waste iterations on badly scaled directions.</font>

### Q3.2 — Evaluate the model
Compute the accuracy and per-class F1 and macro-average F1 on the **test set**, and visualise the confusion matrix.

<font color='red'>
- Prevalent classes are often easier to classify because the model sees more examples.
- However, prevalence is not the only factor: some rare classes can still be easy if they are well-separated in feature space, whereas some frequent classes remain confused if they overlap strongly.
</font>

In [ ]:
# Q3.1
clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=250,
        n_jobs=-1
    )
)

clf.fit(X_train, y_train)

In [ ]:
# Q3.2
def eval_clf(clf, X_test, y_test, silent=False):


    y_test_pred = clf.predict(X_test)
    test_acc = accuracy_score(y_test, y_test_pred)

    # Macro F1
    f1 = f1_score(y_test, y_test_pred, average="macro")

    if not silent:
      print("Performance")
      print("----------------")
      print(f"Macro F1: {100*f1:.2f}%")
      print(f"Overall accuracy: {100*test_acc:.2f}%")

      # Per-class F1 (test)
      f1_per_class = f1_score(y_test, y_test_pred, average=None)

      print("\nPer-class F1 (test) and training prevalence")
      print("-" * 43)
      header_name = "Class name" if class_names is not None else "Class"
      print(f"{header_name:>20} | {'F1 (test)':>9} | {'Train %':>8}")
      print("-" * 43)

      for k in range(K):
          name = class_names[k] if class_names is not None else str(k)
          print(f"{name:>20} | {100*f1_per_class[k]:>8.2f}% | {100*train_prevalence[k]:>7.2f}%")

    return f1

f1 = eval_clf(clf, X_test, y_test)

In [ ]:
# Q3.2
cm = confusion_matrix(y_test, clf.predict(X_test))
disp = ConfusionMatrixDisplay(cm, display_labels=class_names if "class_names" in globals() else None)

fig, ax = plt.subplots(figsize=(7, 7))
disp.plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
plt.title("Confusion Matrix (Test)")
plt.show()

## Q4 — Feature Importance (Logistic Regression)

We now analyse which features the logistic regression relies on.

### Q4.1 — Coefficient magnitude baseline
A simple baseline is to rank features by the sum of absolute coefficients across classes:
$$
\text{score}(j) = \sum_{k=1}^{K} |w_{k,j}|.
$$
Why is this only meaningful after normalization?
<font color='red'>
Some variables like elevation appear much more important than others. Noramlizaiton is important as "smaller" variables will have "larger" coeficients regardless of actual importance.
</font>

### Q4.2 — Permutation importance
A stronger, model-agnostic approach is **permutation importance**:
we randomly permute one feature column (destroying its information) and measure the performance drop.

### Q4.3 — Reduced-feature model
Retrain the classifier using only the **top-5 most important features** and compare performance. Is this useful? Why or why not?
<font color='red'>Not really, useless variables are just not used by the regression</font>

### Q4.4 — Intercept analysis
Inspect the intercept vector. What does it tell you about class prevalence / baseline bias?
<font color='red'>
Intercept reflect class prevalence: high value for common classes.
</font>

In [ ]:
#Q4.1
logreg = clf.named_steps["logisticregression"]
coef = logreg.coef_          # shape (K, d)
intercept = logreg.intercept_  # shape (K,)

print("Logistic regression parameters")
print("------------------------------")
print("Coefficient matrix shape:", coef.shape)
print("Intercept shape:", intercept.shape)

# Baseline importance: sum of abs coefficients across classes
feat_score_L1 = np.abs(coef).sum(axis=0)  # shape (d,)
order = np.argsort(-feat_score_L1)

print("\nFeature ranking by sum_k |coef[k, j]|")
print("-------------------------------------")
for idx in order:
    print(f"{feature_names[idx]:>35} : score = {feat_score_L1[idx]:.4f}")


In [ ]:
#Q4.2
result = permutation_importance(
    clf, X_val, y_val,
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

imp = result.importances_mean
imp_std = result.importances_std
order_perm = np.argsort(-imp)

print("Permutation importance (validation)")
print("-"*50)
for idx in order_perm:
    print(f"{feature_names[idx]:>20} : {imp[idx]:.6f} ± {imp_std[idx]:.6f}")


In [ ]:
#Q4.3
top5_idx = order_perm[:5]
top5_names = [feature_names[i] for i in top5_idx]

print("Top-5 selected features:")
for name in top5_names:
    print(" -", name)

# Slice data
X_train_top5 = X_train[:, top5_idx]
X_val_top5   = X_val[:, top5_idx]
X_test_top5  = X_test[:, top5_idx]

clf_top5 = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=250,
        n_jobs=-1,
    )
)

clf_top5.fit(X_train_top5, y_train)

f1 = eval_clf(clf_top5, X_test_top5, y_test)


In [ ]:
#Q4.4
print("Intercept analysis (multinomial logits)")
print("-" * 50)

K = intercept.shape[0]
train_counts = np.bincount(y_train, minlength=K)
train_prevalence = train_counts / train_counts.sum()

header_name = "Class name" if "class_names" in globals() else "Class"
print(f"{header_name:>20} | {'Intercept':>10} | {'Train %':>8}")
print("-" * 50)

for k in range(K):
    name = class_names[k] if "class_names" in globals() else str(k)
    print(f"{name:>20} | {intercept[k]:>10.4f} | {100*train_prevalence[k]:>7.2f}%")

### Q5 — Ridge Regularisation for Logistic Regression
<font color='green'>
AT HOME QUESTION: SKIP DURING TP UNTIL YOU ARE FINISHED WITH THE GUIDED QUESTIONS
</font>

In this question, you will tune the regularisation strength of logistic regression using the validation set, then report the final performance on the test set.

Recall that in scikit-learn’s LogisticRegression, L2 (ridge) regularisation is controlled by the parameter C, where:
 - smaller C → stronger regularisation
 - larger C → weaker regularisation

Q5.1 — Model selection on the validation set

Train several L2-regularised logistic regression models for a range of C values.
For each model:

- train on the training set,

- evaluate on the validation set using macro-F1,

Select the C that maximises validation macro-F1.

Q5.2 — Final evaluation on the test set

Retrain the model on train + validation using the best C, and evaluate once on the test set.

## Q6 — A Single Decision Tree (CART)

We now move from linear models to **decision trees**.  
A decision tree partitions the feature space using a sequence of axis-aligned splits, leading to a piecewise-constant prediction rule.

### Q6.1 — Train a single decision tree
Train a decision tree classifier on the training set.  
Evaluate it on both the training and validation sets.
How does this model compare to logistic regression? Why?
<font color='red'>
Not linearly seperable, so trees are a better fit. Also, a lot of different vraiable types, better with trees. Tree-based methods do not require feature scaling, as splits depend on orderings, not magnitudes
</font>

### Q6.2 — Visualise the tree structure (small depth)
To make the model interpretable, train a shallow tree (e.g. `max_depth=3`) and visualise it:
- Which features appear near the root?
- What does it mean when a feature is used near the top?
<font color='red'>
Elevation is the most important, it is very discriminative
</font>

### Q6.3 — Overfitting check
Train a deeper tree (e.g. `max_depth=None` or a large value) and compare train vs validation metrics.
- What do you observe?
- Why are trees prone to overfitting?
<font color='red'>
The tree overfits, but still perform well on the validation. Whil yes, single tree can overfit, they do not always do. In particualr when the features are well designed. Note that infinite depth does not guarantee full memorisation. Tree growth may stop when no split improves impurity.
</font>

In [ ]:
#Q6.1
tree = DecisionTreeClassifier(
    random_state=42
)
tree.fit(X_train, y_train)

f1 = eval_clf(tree, X_test, y_test)

In [ ]:
#Q6.2
tree_small = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)
tree_small.fit(X_train, y_train)

f1 = eval_clf(tree_small, X_test, y_test)

plt.figure(figsize=(30, 8))
plot_tree(
    tree_small,
    feature_names=(feature_names if "feature_names" in globals() else None),
    class_names=(class_names if "class_names" in globals() else None),
    filled=True,
    rounded=True,
    impurity=True,
    fontsize=10
)
plt.title("Decision Tree (max_depth=3)")
plt.show()

In [ ]:
#Q6.3
tree_deep = DecisionTreeClassifier(
    max_depth=None,   # fully grown
    random_state=42
)
tree_deep.fit(X_train, y_train)

res_tree_deep = f1 = eval_clf(tree_deep, X_train, y_train)

res_tree_deep = f1 = eval_clf(tree_deep, X_test, y_test)

## Q7 — Random Forests

Random forests combine many decision trees to reduce variance by averaging de-correlated predictors.

### Q7.1 — Train a random forest
Train a random forest classifier and evaluate it on train / test.

### Q7.2 — Effect of tree depth
Study how `max_depth` affects performance:
- Train several forests with different `max_depth`.
- Plot train and validation macro-F1 as a function of max depth.
- At what depth do you observe overfitting/saturation?
<font color=red> beyond 20, the benefits are small. Does not decrease, thanks to the robustness of RF. </font>

### Q7.3 — Effect of the number of trees
Study how the number of trees (`n_estimators`) affects performance:
- Train forests with increasing `n_estimators`.
- Plot train and validation macro-F1 as a function of `n_estimators`.
- Does validation performance saturate? Why?

DecisionTreeClassifier

### Q7.4 — Final Model
Train the best configuration on train + val and evaluate on test.

In [ ]:
#Q7.1
rf = RandomForestClassifier(
    n_estimators=50,
    random_state=42,
    n_jobs=-1,
    max_depth=None
)
rf.fit(X_train, y_train)

f1 = eval_clf(rf, X_test, y_test)

In [ ]:
#Q7.2
depth_grid = [2, 3, 5, 8, 12, 20, None]
train_scores = []
val_scores = []

for d in depth_grid:
    rf_d = RandomForestClassifier(
        n_estimators=50,
        max_depth=d,
        random_state=42,
        n_jobs=-1
    )
    rf_d.fit(X_train, y_train)

    y_tr_pred = rf_d.predict(X_train)
    y_va_pred = rf_d.predict(X_val)

    train_scores.append(f1_score(y_train, y_tr_pred, average="macro"))
    val_scores.append(f1_score(y_val, y_va_pred, average="macro"))

plt.figure()
x = list(range(len(depth_grid)))
plt.plot(x, train_scores, marker="o", label="Train macro-F1")
plt.plot(x, val_scores, marker="o", label="Val macro-F1")
plt.xticks(x, [str(d) for d in depth_grid])
plt.xlabel("max_depth")
plt.ylabel("macro-F1")
plt.title("Random Forest: effect of max_depth")
plt.grid(True, linestyle="--", linewidth=0.5)
plt.legend()
plt.show()

In [ ]:
#Q7.3
n_grid = [10, 25, 50, 100, 200, 400]
train_scores = []
val_scores = []

for n_estimators in n_grid:
    rf_n = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    )
    rf_n.fit(X_train, y_train)

    y_tr_pred = rf_n.predict(X_train)
    y_va_pred = rf_n.predict(X_val)

    train_scores.append(f1_score(y_train, y_tr_pred, average="macro"))
    val_scores.append(f1_score(y_val, y_va_pred, average="macro"))

plt.figure()
plt.plot(n_grid, train_scores, marker="o", label="Train macro-F1")
plt.plot(n_grid, val_scores, marker="o", label="Val macro-F1")
plt.xlabel("n_estimators")
plt.ylabel("macro-F1")
plt.title("Random Forest: effect of number of trees")
plt.grid(True, linestyle="--", linewidth=0.5)
plt.legend()
plt.show()


In [ ]:
#Q7.4
best_rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)
best_rf.fit(X_trainval, y_trainval)
res_best_rf = eval_clf(best_rf, X_test, y_test)


## Q8 — Boosting (Gradient Boosted Decision Trees)

<font color='green'>
AT HOME QUESTION: SKIP DURING TP UNTIL YOU ARE FINISHED WITH THE GUIDED QUESTIONS
</font>

We now study **boosting**, which builds an ensemble *sequentially*: each new tree corrects the errors of the current model.
In practice, gradient boosting often outperforms random forests on tabular datasets, but it is more sensitive to hyperparameters.

We will use a modern implementation of gradient boosting trees: `HistGradientBoostingClassifier`.

### Q8.1 — Train a baseline boosted tree model
Train a gradient boosting model and evaluate it on train / validation.

### Q8.2 — Effect of tree complexity (max_depth)
Study how `max_depth` affects performance:
- Train models for several values of `max_depth`
- Plot train and validation macro-F1 as a function of `max_depth`
- Identify underfitting vs overfitting behaviour

### Q8.3 — Effect of the learning rate
Study how `learning_rate` affects performance:
- Train models for several values of `learning_rate`
- Plot train and validation macro-F1 vs learning rate
- Explain why a smaller learning rate often requires more boosting iterations

### Q8.4 — Compare to random forests
Compare the boosted model and the random forest model:
- Which one performs best on validation macro-F1?
- Which one is more sensitive to hyperparameters?
- Which one is faster to train in your environment?


### Dataset courtesy of:

Remote Sensing and GIS Program

Department of Forest Sciences

College of Natural Resources

Colorado State University

Fort Collins, CO 80523